# Migrate from Secondary EdgeDNS zone setup to Primary

When you have EdgeDNS running in Secondary mode, it's pretty simple to convert a zone, or list of secondary zones, to a primary setup. During the conversion, existing records will be made editable so EdgeDNS becomes the primary; no need for the secondary setup anymore.

There is an EdgeDNS API call that can be used to [convert a zone from secondary to primary](https://techdocs.akamai.com/edge-dns/reference/post-zones-convert-primary-requests). No need to export and import DNS records yourself, the API call will do this for you.

Before trying to convert a list of zones, the notebook makes sure the mode is SECONDARY, and the activation state is ACTIVE (or ERROR). Doesn't make much sense to convert a primary to primary. ;-)

> **Using DNSSEC on your hidden primary?** The zone will temporarily go unsigned (insecure) during the migration, as the hidden primary's keys can't be moved to EdgeDNS. Follow this order:
> 1. Remove the DS record for the zone at your registrar.
> 2. Wait until the DS record is gone from the parent zone's name servers, **plus** the DS record's TTL, so validating resolvers have dropped it from their cache. Disabling DNSSEC before that makes the zone fail validation (SERVFAIL).
> 3. Disable DNSSEC on the hidden primary, bump the SOA serial and make sure EdgeDNS has transferred the unsigned zone (no more DNSKEY/RRSIG records served by EdgeDNS).
> 4. Convert the zone to primary and enable DNSSEC Sign and Serve on EdgeDNS.
> 5. Submit the new DS record generated by EdgeDNS to your registrar.
>
> If an unsigned window is not acceptable, a multi-signer migration ([RFC 8901](https://www.rfc-editor.org/rfc/rfc8901)) is the alternative, but that is out of scope for this notebook.

## Set Up EdgeGrid Session

Let's first set up an EdgeGrid session using information from the `~/.edgerc` file. The correct *section* can be set via `export AKAMAI_EDGEGRID_SECTION=section`; otherwise, `default` is going to be used.

Make sure your API user has the [EdgeDNS permissions](https://techdocs.akamai.com/edge-dns/reference/get-started).

This notebook supports the `AccountSwitchKey` option which can be set via `export AKAMAI_ACCOUNT_SWITCH_KEY="1-2345:A-BCDE"`

In [1]:
import json
import os

import requests
from akamai.edgegrid import EdgeGridAuth, EdgeRc

edgerc_path = os.path.expanduser("~/.edgerc")

if not os.path.exists(edgerc_path):
    raise FileNotFoundError(f"Could not find .edgerc at {edgerc_path}")

section = os.getenv("AKAMAI_EDGEGRID_SECTION", "default")
account_switch_key = os.getenv("AKAMAI_ACCOUNT_SWITCH_KEY")

edgerc = EdgeRc(edgerc_path)
host = edgerc.get(section, "host")

session = requests.Session()
session.auth = EdgeGridAuth.from_edgerc(edgerc, section)
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
})

base_url = f"https://{host}"
print(f"EdgeGrid session configured for host: {host}")

if account_switch_key:
    print(f"Using AccountSwitchKey: {account_switch_key}")

EdgeGrid session configured for host: akab-jdn2ct6ewvfefhhb-csofinvorrtwiu5w.luna.akamaiapis.net
Using AccountSwitchKey: B-M-1YX7F48:1-8BYUX


## Validate state of zones

Doesn't make much sense to try to convert a primary zone to primary. The zone must be SECONDARY with an activation state of ACTIVE or ERROR.

In [2]:
zone_names = ["nipshotnews.nl"]
eligible_activation_states = {"ACTIVE", "ERROR"}
params = {"accountSwitchKey": account_switch_key} if account_switch_key else None

for zone_name in zone_names:
    zone_url = f"{base_url}/config-dns/v2/zones/{zone_name}"
    zone_response = session.get(zone_url, params=params)
    zone_response.raise_for_status()

    zone_details = zone_response.json()
    zone_type = zone_details["type"]
    activation_state = zone_details["activationState"]
    print(f"{zone_name}: type={zone_type}, activationState={activation_state}")

    if zone_type != "SECONDARY":
        raise RuntimeError(
            f"{zone_name} is a {zone_type} zone and is not eligible for conversion. "
            "Only SECONDARY zones can be converted to PRIMARY."
        )

    if activation_state not in eligible_activation_states:
        raise RuntimeError(
            f"{zone_name} activation state {activation_state} is not eligible for conversion. "
            f"The activation state must be one of {sorted(eligible_activation_states)}."
        )

nipshotnews.nl: type=SECONDARY, activationState=ACTIVE


## Check if DNSSEC is still enabled

EdgeDNS is going to sign the zone with new key material, so the old DNSSEC setup must be fully gone before converting. Two checks per zone:

1. **No DS record in the parent zone.** The parent zone's authoritative name servers are queried directly, so we don't rely only on a cached answer from a resolver. An empty answer here doesn't mean resolvers have dropped the DS yet; wait at least the old DS TTL after removing it.
2. **No DNSKEY record served by EdgeDNS.** This confirms DNSSEC is disabled on the hidden primary and EdgeDNS has transferred the unsigned zone.

Both checks also query Google Public DNS, as some networks intercept DNS traffic and answer "direct" queries from a resolver. A warning is printed when an answer is not authoritative. `+cd` (checking disabled) is used so a zone that is already failing validation still returns its records instead of an empty SERVFAIL answer.

In [3]:
import re
import shutil
import subprocess

if shutil.which("dig") is None:
    raise RuntimeError("The DNSSEC delegation check requires the dig command to be installed.")

google_dns_resolver = "8.8.8.8"


def dig(server, record_type, name, recurse=True):
    """Query server and return the answer records of record_type, failing on any non-NOERROR status."""
    command = ["dig", f"@{server}", "+cd", "+noall", "+comments", "+answer", record_type, name]
    if not recurse:
        command.insert(2, "+norec")
    result = subprocess.run(command, capture_output=True, text=True, check=False)

    status = re.search(r"status: (\w+)", result.stdout)
    if result.returncode != 0 or status is None or status.group(1) != "NOERROR":
        raise RuntimeError(
            f"{record_type} query for {name} at {server} failed: "
            f"{status.group(1) if status else result.stderr.strip() or 'no response'}"
        )

    # Some networks intercept DNS traffic, so a "direct" query may be answered by a resolver instead.
    flags = re.search(r"flags: ([\w ]+);", result.stdout)
    if not recurse and (flags is None or "aa" not in flags.group(1).split()):
        print(f"WARNING: {record_type} answer for {name} from {server} is not authoritative, DNS may be intercepted")

    return [
        line.split()
        for line in result.stdout.splitlines()
        if line and not line.startswith(";") and line.split()[3] == record_type
    ]


def parent_name_server(zone_name):
    """Walk up the labels of zone_name until we find the parent zone cut and return one of its name servers."""
    parent = zone_name
    while "." in parent:
        parent = parent.split(".", 1)[1]
        ns_records = dig(google_dns_resolver, "NS", parent)
        if ns_records:
            return parent, ns_records[0][4]
    raise RuntimeError(f"Could not find the parent zone of {zone_name}")


for zone_name in zone_names:
    parent, parent_ns = parent_name_server(zone_name)
    ds_records = dig(parent_ns, "DS", zone_name, recurse=False) + dig(google_dns_resolver, "DS", zone_name)
    print(f"{zone_name}: {len(ds_records)} DS record(s) from parent zone {parent} ({parent_ns}) and {google_dns_resolver}")

    if ds_records:
        raise RuntimeError(
            f"{zone_name} still has DS record(s) in the parent zone (TTL {ds_records[0][1]}s). "
            "Remove the DS record at your registrar and wait for this TTL to expire, "
            "then disable DNSSEC on the hidden primary before converting."
        )

    zone_ns_records = dig(google_dns_resolver, "NS", zone_name)
    if not zone_ns_records:
        raise RuntimeError(f"Could not find the name servers of {zone_name}")
    zone_ns = zone_ns_records[0][4]

    dnskey_records = dig(zone_ns, "DNSKEY", zone_name, recurse=False) + dig(google_dns_resolver, "DNSKEY", zone_name)
    print(f"{zone_name}: {len(dnskey_records)} DNSKEY record(s) from {zone_ns} and {google_dns_resolver}")

    if dnskey_records:
        raise RuntimeError(
            f"{zone_name} is still signed on {zone_ns}. Disable DNSSEC on the hidden primary, "
            "bump the SOA serial and wait for EdgeDNS to transfer the unsigned zone before converting."
        )

nipshotnews.nl: 0 DS record(s) from parent zone nl (ns3.dns.nl.) and 8.8.8.8
nipshotnews.nl: 0 DNSKEY record(s) from c2-41.akashield.net. and 8.8.8.8


## Convert zone

Now all zones in `zone_names` are converted from secondary to primary in a single convert request and off you go. 

In [4]:
zones_to_convert = {
    "comment": "Convert secondary zone(s) to primary",
    "zoneList": [{"name": zone_name} for zone_name in zone_names],
}

convert_primary_url = f"{base_url}/config-dns/v2/zones/convert-requests/primary"
params = {"accountSwitchKey": account_switch_key} if account_switch_key else None

response = session.post(
    convert_primary_url,
    params=params,
    json=zones_to_convert,
)

print(f"HTTP {response.status_code}")
print(json.dumps(response.json(), indent=2))
response.raise_for_status()

HTTP 200
{
  "requestId": "b0dca57c-deee-423e-a238-78e4798479c5",
  "zonesSubmitted": 1,
  "successCount": 0,
  "failureCount": 0,
  "isComplete": false,
  "expirationDate": "2026-10-22T07:25:50.839442Z"
}


## Enable DNSSEC Sign and Serve

Once the zone is an ACTIVE primary, enable DNSSEC Sign and Serve so EdgeDNS generates new keys and signs the zone. The conversion can take a moment; rerun this cell if the zone is not ACTIVE yet.

The update payload is built from the current zone settings, keeping only the writable fields, so no existing settings are reset.

In [5]:
sign_and_serve_algorithm = "ECDSA_P256_SHA256"
writable_zone_fields = {
    "zone", "type", "signAndServe", "signAndServeAlgorithm", "comment", "contractId",
    "endCustomerId", "masters", "target", "subzoneGrants", "tsigKey",
    "multiProviderDnssec", "outboundZoneTransfer",
}

for zone_name in zone_names:
    zone_url = f"{base_url}/config-dns/v2/zones/{zone_name}"
    zone_response = session.get(zone_url, params=params)
    zone_response.raise_for_status()

    zone_details = zone_response.json()
    zone_type = zone_details["type"]
    activation_state = zone_details["activationState"]
    print(f"{zone_name}: type={zone_type}, activationState={activation_state}")

    if zone_type != "PRIMARY" or activation_state != "ACTIVE":
        raise RuntimeError(
            f"{zone_name} is not ready for DNSSEC. "
            f"Expected type PRIMARY and activationState ACTIVE, got "
            f"type {zone_type} and activationState {activation_state}."
        )

    if zone_details.get("signAndServe", False):
        print(f"{zone_name}: DNSSEC Sign and Serve is already enabled")
        continue

    zone_settings = {key: value for key, value in zone_details.items() if key in writable_zone_fields}
    zone_settings["signAndServe"] = True
    zone_settings["signAndServeAlgorithm"] = sign_and_serve_algorithm

    enable_dnssec_response = session.put(zone_url, params=params, json=zone_settings)

    print(f"{zone_name}: HTTP {enable_dnssec_response.status_code} enabling DNSSEC")
    print(json.dumps(enable_dnssec_response.json(), indent=2))
    enable_dnssec_response.raise_for_status()

nipshotnews.nl: type=PRIMARY, activationState=PENDING


RuntimeError: nipshotnews.nl is not ready for DNSSEC. Expected type PRIMARY and activationState ACTIVE, got type PRIMARY and activationState PENDING.

## Get the new DS record

Final step: fetch the DS record generated by EdgeDNS via the [DNSSEC status](https://techdocs.akamai.com/edge-dns/reference/post-zones-dns-sec-status) API and submit it to your registrar. Key generation can take a moment; rerun this cell if no DS record is returned yet.

Until the DS record is published in the parent zone, the API reports a `PARENT_DS_MISSING` alert. Rerun this cell after submitting the DS record to confirm it's gone.

In [ ]:
dnssec_status_url = f"{base_url}/config-dns/v2/zones/dns-sec-status"

dnssec_status_response = session.post(
    dnssec_status_url,
    params=params,
    json={"zones": zone_names},
)
dnssec_status_response.raise_for_status()

for dnssec_status in dnssec_status_response.json()["dnsSecStatuses"]:
    zone_name = dnssec_status["zone"]
    ds_record = dnssec_status["currentRecords"].get("dsRecord")

    if not ds_record:
        print(f"{zone_name}: no DS record generated yet, rerun this cell in a moment")
        continue

    print(f"{zone_name}: alerts={dnssec_status['alerts']}")
    print(f"{zone_name}: DS record to submit to your registrar:\n{ds_record}\n")